# Verse similarity

K-means uses normalized proportions. Inspect silhouette and seed stability; clusters are not genres.

Computational logic is in `src/torah_cantillation/`. These notebooks inspect generated evidence, not live API responses.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SNAPSHOT = 'primary-20261013'
REPORT = ROOT / "reports" / SNAPSHOT
assert (REPORT / "summary.json").exists(), "Run fetch, build and analyze for this snapshot first."
summary = json.loads((REPORT / "summary.json").read_text())
display(Markdown("**PILOT ONLY**" if summary["pilot"] else "Validated full reference coverage"))
print("Snapshot:", summary["snapshot"])


Validated full reference coverage

Snapshot: primary-20261013


## Cluster Diagnostics

In [2]:
values = json.loads((REPORT / "tables" / "cluster_diagnostics.json").read_text())
frame = pd.DataFrame(values if isinstance(values, list) else [values])
print("Total rows:", len(frame))
display(frame.head(40))

Total rows: 5


,edition_id,features,k,seed_stability_adjusted_rand,silhouette,status
0,tanach,within-verse token-distinct accent proportions,2,0.996382,0.313424,computed
1,tanach,within-verse token-distinct accent proportions,3,0.932126,0.207684,computed
2,tanach,within-verse token-distinct accent proportions,4,0.682044,0.178025,computed
3,tanach,within-verse token-distinct accent proportions,5,0.995059,0.161131,computed
4,tanach,within-verse token-distinct accent proportions,6,0.711099,0.154449,computed


## Clusters

In [3]:
values = json.loads((REPORT / "tables" / "clusters.json").read_text())
frame = pd.DataFrame(values if isinstance(values, list) else [values])
print("Total rows:", len(frame))
display(frame.head(40))

Total rows: 4686


,book,cluster,edition_id,pca_1,pca_2,reference,selected_k
0,Genesis,0,tanach,0.210830,0.052791,Genesis 1:1,2
1,Genesis,1,tanach,-0.056195,-0.020480,Genesis 1:2,2
2,Genesis,0,tanach,0.095957,0.066158,Genesis 1:3,2
3,Genesis,0,tanach,0.080122,-0.016164,Genesis 1:4,2
4,Genesis,1,tanach,-0.052804,0.006010,Genesis 1:6,2
5,Genesis,1,tanach,-0.279788,-0.034998,Genesis 1:7,2
6,Genesis,1,tanach,-0.224762,-0.025627,Genesis 1:9,2
7,Genesis,1,tanach,0.007201,-0.025659,Genesis 1:10,2
8,Genesis,1,tanach,-0.219050,-0.071194,Genesis 1:11,2
9,Genesis,1,tanach,-0.023435,-0.070244,Genesis 1:12,2


## Interpretation

Check denominators, exclusions and edition scope. A truncated notebook display is not the complete occurrence catalog. See `summary.json` for checksums and environment versions.

In [4]:
display(Markdown((REPORT / "findings.md").read_text()))

# Computed findings

Snapshot: `primary-20261013`. **Full selected reference coverage validated.**

Encoded accents U+0591-U+05AE; all_signs separately audits U+05BD and other signs

## Corpus inventory

- `tanach`: 5,846 verses; 80,742 lexical tokens; 64,234 encoded accent signs in 26 code points; 4,686 verses eligible for sequence analysis.
  Most frequent encoded accents: HEBREW ACCENT TIPEHA (11,290); HEBREW ACCENT MUNAH (9,462); HEBREW ACCENT MERKHA (9,120); HEBREW ACCENT ZAQEF QATAN (6,998); HEBREW ACCENT PASHTA (6,378).

## Sequential dependence

- `tanach` / `literal`: observed adjacent-mark mutual information 1.6635 bits; within-verse shuffle mean 0.0562 bits; exploratory p=0.01; Bonferroni-adjusted p=0.02 (99 shuffles).
- `tanach` / `token_distinct`: observed adjacent-mark mutual information 1.6950 bits; within-verse shuffle mean 0.0576 bits; exploratory p=0.01; Bonferroni-adjusted p=0.02 (99 shuffles).

## Predictability

Leave-one-book-out log losses are in `tables/model_evaluation.json`. Unigram and bigram baselines use add-one smoothing and a fixed 30-symbol Unicode alphabet. Repeated consonantal verse texts in held-out books are removed from training.

## Interpretation limits

- Counts describe the encoded source snapshot, not every manuscript or chanting tradition.
- Literal inventory includes text in unresolved readings; inspect quality flags before claiming lexical or linguistic counts.
- Repeated signs remain in the literal layer. Token-distinct collapse is a sensitivity representation, not a linguistic truth.
- Markup, bracketed readings, orphan accents, multiple distinct accents on a token, and nonstandard verse endings exclude verses from sequence/model analysis, not the raw audit.
- Clusters and high-surprise verses are exploratory, not inferred genres, textual errors, or historical explanations.
- The shuffle tests one explicit null model. Descriptive counts are a corpus census, not sample estimates.
- Edition comparison is unrun unless two editions were acquired; inspect its status table.
- Silluq candidates are audited separately and excluded from accent-codepoint sequences.

## Evidence

All tables are under `tables/`. `summary.json` records provenance, environment versions, code and table checksums. `rare_occurrences.json` lists every occurrence of an encoded accent with total frequency <=20 in an edition. The complete occurrence inventory, including common signs, is `data/processed/<snapshot>/marks.jsonl`.
